# Aerosol Tutorial

An aerosol holds three native containers: particle material and population,
ordered gas species, and environment. Physics configuration stays separate.
Here we initialize glycerol at half its saturation concentration and a
lognormal particle population. Construction retains the supplied objects.

In [1]:
import numpy as np
import particula as par

## Glycerol thermodynamics

A vapor-pressure strategy computes the initial saturation concentration.
It is not stored in gas data; a subsequent process owns its configuration.

In [2]:
molar_mass_glycerol = 92.09382e-3  # kg/mol
parameters_clausius = {
    "latent_heat": 71.5 * molar_mass_glycerol,
    "latent_heat_units": "J/mol",
    "temperature_initial": 125.5,
    "temperature_initial_units": "degC",
    "pressure_initial": 1,
    "pressure_initial_units": "mmHg",
}
vapor_pressure_strategy = par.gas.VaporPressureFactory().get_strategy(
    "clausius_clapeyron", parameters_clausius
)

sat_concentration = vapor_pressure_strategy.saturation_concentration(
    molar_mass_glycerol, 298.15
)
print(f"Saturation concentration: {sat_concentration:.2e} kg/m^3")

sat_factor = 0.5
glycerol_gas = (
    par.gas.GasDataBuilder()
    .set_names(["Glycerol"])
    .set_molar_mass([molar_mass_glycerol], "kg/mol")
    .set_concentration([sat_concentration * sat_factor], "kg/m^3")
    .set_partitioning([True])
    .build()
)
environment = par.gas.EnvironmentData(
    temperature=np.array([298.15]),
    pressure=np.array([101325.0]),
    saturation_ratio=np.array([[sat_factor]]),
)

Saturation concentration: 4.95e-03 kg/m^3


## Native particle initialization

Discrete bins store number density in m^-3 at reference volume 1 m^3.
Each mass lane is per-particle mass in kg, not population mass. The preset
retains the original 100 nm mode, geometric spread, and number density.

In [3]:
particles = par.particles.create_lognormal_binned_particle_data(
    distribution_type="discrete",
    mode=[100e-9],
    geometric_standard_deviation=[1.5],
    number_concentration=[1e10],  # 1e4 cm^-3
    density=1260.0,  # 1.26 g/cm^3
)

## Assemble and inspect the held state

Partitioning flags distinguish condensable species from inert gas lanes in
one ordered gas container. No species list is filtered during construction.

In [4]:
aerosol = par.Aerosol(
    particles=particles, gas=glycerol_gas, environment=environment
)
assert aerosol.particles is particles
assert aerosol.gas is glycerol_gas
assert aerosol.environment is environment
print(aerosol)
print("Number density [m^-3]:", particles.concentration.sum(axis=1))
print(
    "Particle mass density [kg/m^3]:",
    np.sum(particles.masses * particles.concentration[:, :, None], axis=1),
)

ParticleData(masses=array([[[5.27787566e-24],
        [6.06317521e-24],
        [6.96532014e-24],
        [8.00169597e-24],
        [9.19227503e-24],
        [1.05600013e-23],
        [1.21312328e-23],
        [1.39362491e-23],
        [1.60098353e-23],
        [1.83919522e-23],
        [2.11285062e-23],
        [2.42722344e-23],
        [2.78837205e-23],
        [3.20325627e-23],
        [3.67987146e-23],
        [4.22740262e-23],
        [4.85640141e-23],
        [5.57898945e-23],
        [6.40909197e-23],
        [7.36270615e-23],
        [8.45820938e-23],
        [9.71671345e-23],
        [1.11624714e-22],
        [1.28233449e-22],
        [1.47313411e-22],
        [1.69232298e-22],
        [1.94412513e-22],
        [2.23339314e-22],
        [2.56570158e-22],
        [2.94745448e-22],
        [3.38600870e-22],
        [3.88981578e-22],
        [4.46858474e-22],
        [5.13346921e-22],
        [5.89728241e-22],
        [6.77474401e-22],
        [7.78276385e-22],
        [8.9407678

The aggregate exposes its held containers through properties. Mutating their
arrays changes the held state; use `copy()` for detached inspection. Processes
validate their own scientific requirements before using this state.